# 02 - Bias search: which attribute should we mitigate?

**What you'll learn**
- How `BiasSearch` ranks several sensitive attributes by how biased a model is for each
- How to read the result table (raw vs normalized score, group sizes)
- How to study intersections of attributes (e.g. group A *and* group B)
- How set operations (union, difference, ...) define new groups
- Why very small groups give unreliable scores

**Prerequisite:** [01_fairness_fundamentals.ipynb](01_fairness_fundamentals.ipynb) (metrics and the fairness score). **Runtime:** about 2 minutes.

## 1. The question

A dataset often has several sensitive attributes. Bias mitigation is costly and can reduce accuracy, so we first ask:
**for which attribute is the model most biased?** `BiasSearch` answers this by repeating, for every attribute, the pipeline of notebook 01
(split, train, choose threshold on validation, compute the fairness score on test) and averaging the score over several random splits.

We use the *WiDS 2024* data (label `DiagPeriodL90D` = 1: metastatic cancer diagnosed within 90 days) with its five binary sensitive attributes.

In [1]:
from fhg_examples import setup
setup()

import matplotlib.pyplot as plt
import pandas as pd

from callmefair import BiasSearch, CType, combine_attributes
from fhg_examples.datasets import load, SPECS

In [2]:
df = load('wids')
spec = SPECS['wids']
attributes = spec.sensitive_names
print(df.shape, attributes)
df[attributes].agg(['sum']).rename(index={'sum': 'rows in group 1'}).assign(total_rows=len(df))

(12823, 21) ['patient_age', 'race', 'payer', 'state_privileged', 'neighborhood_education']


,patient_age,race,payer,state_privileged,neighborhood_education,total_rows
rows in group 1,105,9913,10279,4662,6377,12823


In every attribute, **1 is the privileged and 0 the unprivileged group** (see `00_data_and_preprocessing.ipynb` for what each value means). Note that `patient_age` has only 105 rows in group 1; we come back to this in section 5.

## 2. Search over single attributes

`evaluate_average` runs `iterate` splits per attribute (each with a different seed) and averages the scores.
Here we use 5 iterations and a logistic regression (`model_name='lr'`) to keep it quick; use 10 or more for real studies.
`progress=False` hides the progress bars. Iterations run in parallel worker processes.
It returns the table as a list of rows (header first) and a pretty-printed version; we turn the list into a DataFrame.

In [3]:
search = BiasSearch(df, label_name=spec.label, attribute_names=attributes, progress=False)
table, _ = search.evaluate_average(iterate=5, model_name='lr')
results = pd.DataFrame(table[1:], columns=table[0]).set_index('Attribute')
results.sort_values('Normalized Fairness score', ascending=False)

,Raw Fairness Score,Normalized Fairness score,Balanced Accuracy,Privileged N,Unprivileged N
Attribute,,,,,
patient_age,1.060,0.589,0.761,105,12718
state_privileged,0.173,0.096,0.754,4662,8161
race,0.160,0.089,0.753,9913,2910
neighborhood_education,0.144,0.080,0.758,6377,6446
payer,0.133,0.074,0.758,10279,2544


How to read it (sorted from most to least biased):
- **Normalized Fairness score** in [0, 1]: the fairness score of notebook 01 (0 = fair, higher = more biased). Use this to compare attributes.
- **Raw Fairness Score**: the same score *before* dividing by its maximum and clipping to [0, 1]; it keeps the magnitude when several attributes are all very biased.
- **Balanced Accuracy**: model quality for that run, to check that a "fair" model is not simply a useless one.
- **Privileged N / Unprivileged N**: group sizes in the whole dataset.

In [ ]:
from callmefair.viz import plot_bias_search
plot_bias_search(table)

`patient_age` ranks first by a wide margin, but its group 1 has only 105 rows, so this score is the least reliable one (see section 5).

## 3. Intersections of attributes

A model may look fair for each attribute alone but not for people in *both* groups at once (for example, group 1 of attribute A **and** group 1 of attribute B).
`evaluate_combinations` evaluates the intersection of every pair and triple of attributes.
Combinations where a group is too small (fewer than 10 rows, or fewer than 2 per label) are skipped and reported.
It defaults to LightGBM; we pass `model_name='lr'` for speed.

In [5]:
comb_table, _ = search.evaluate_combinations(iterate=3, model_name='lr')
comb = pd.DataFrame(comb_table[1:], columns=comb_table[0]).set_index('Attribute')
pd.set_option('display.max_colwidth', 70)
cols = ['Normalized Fairness score', 'Balanced Accuracy', 'Privileged N']
comb.sort_values('Normalized Fairness score', ascending=False)[cols].head(8)
top_comb = comb.sort_values('Normalized Fairness score', ascending=False).head(8).reset_index()

,Normalized Fairness score,Balanced Accuracy,Privileged N
Attribute,,,
patient_age_state_privileged_neighborhood_education,0.989,0.756,41
patient_age_payer_state_privileged,0.968,0.751,32
patient_age_state_privileged,0.907,0.753,62
patient_age_race_payer,0.803,0.757,34
patient_age_race_neighborhood_education,0.791,0.763,28
patient_age_payer_neighborhood_education,0.778,0.753,30
patient_age_payer,0.752,0.752,57
patient_age_race_state_privileged,0.571,0.767,26


An intersection group (value 1) is small by construction, so check `Privileged N` before trusting a score. Read the intersection ranking next to the single-attribute table above.

In [ ]:
plot_bias_search(top_comb)

## 4. Other set operations

`combine_attributes` merges two binary columns into one new binary column; `CType` selects the rule.
With attribute A and B, the new column is 1 when:

| CType | A | B | result |
|---|---|---|---|
| `intersection` | 1 and 1 | | A AND B |
| `union` | | | A OR B |
| `difference_1_minus_2` | | | A AND NOT B |
| `difference_2_minus_1` | | | B AND NOT A |
| `symmetric_difference` | | | A XOR B (exactly one) |

Truth table on all four (A, B) cases:

In [6]:
truth = pd.DataFrame({'A': [0, 0, 1, 1], 'B': [0, 1, 0, 1]})
for op in CType:
    truth[op.name] = combine_attributes(truth[['A', 'B']], 'A', 'B', op)['A_B']
truth

,A,B,union,intersection,difference_1_minus_2,difference_2_minus_1,symmetric_difference
0,0,0,0,0,0,0,0
1,0,1,1,0,0,1,1
2,1,0,1,0,1,0,1
3,1,1,1,1,0,0,0


`evaluate_combination_average` evaluates **all five** operations for one pair of attributes, so you can see which definition of the group is most biased.

In [7]:
pair = ('race', 'payer')
op_table, _ = search.evaluate_combination_average(*pair, iterate=3, model_name='lr')
pd.DataFrame(op_table[1:], columns=op_table[0]).set_index('Operator').drop(columns='Attribute')

,Raw Fairness Score,Normalized Fairness score,Privileged N,Unprivileged N
Operator,,,,
union,0.246,0.137,11719,1104
intersection,0.234,0.130,8473,4350
difference_1_minus_2,0.247,0.137,1440,11383
difference_2_minus_1,0.193,0.107,1806,11017
symmetric_difference,0.185,0.103,3246,9577


## 5. Caveat: small groups

The score of a group with a handful of rows is dominated by chance: a few test patients decide each rate.
We can see this by repeating the evaluation of `patient_age` (105 rows in group 1, so only ~16 in each test split) with different splits and comparing with the large group `race`. Each row below summarises 8 single-split scores; a large `std`/range means the ranking depends on the split.

In [8]:
def one_split_score(attr, seed):
    one = BiasSearch(df, spec.label, [attr], random_state=seed, n_threads=1, progress=False)
    return one.evaluate_attribute(attr, iterate=1, model_name='lr')[f'{attr}_overall']

scores = pd.DataFrame({attr: [one_split_score(attr, seed) for seed in range(8)]
                       for attr in ['patient_age', 'race']})
display(scores.agg(['mean', 'std', 'min', 'max']).T)

import numpy as np
from callmefair.viz._style import new_axes, color, INK_SECONDARY
fig, ax = new_axes(figsize=(5.6, 3.6))
for k, attr in enumerate(scores.columns):
    ax.scatter(np.full(len(scores), k) + np.linspace(-0.08, 0.08, len(scores)), scores[attr], color=color(k), s=28, zorder=3)
    ax.hlines(scores[attr].mean(), k - 0.25, k + 0.25, color=color(k), lw=2, zorder=4)
ax.set_xticks(range(len(scores.columns)), scores.columns)
ax.set_xlim(-0.6, len(scores.columns) - 0.4)
ax.set_ylabel('fairness score of one split\n(0 = fair)')
ax.set_title('Score over 8 different splits (line = mean)')
ax.grid(False, axis='x')
fig

,mean,std,min,max
patient_age,0.732,0.206,0.365,1.000
race,0.120,0.037,0.064,0.184


## Summary
- `BiasSearch.evaluate_average` ranks attributes by the average **normalized fairness score** (0 = fair, higher = more biased); start mitigation with the top one.
- Always read the score with the group sizes and the balanced accuracy next to it.
- `evaluate_combinations` studies intersections; `evaluate_combination_average` compares the five `CType` set operations for a pair.
- Scores of tiny groups are noisy: use more `iterate`, and treat such results with caution.
- `progress=False` keeps notebook output clean.

**Next:** [03_bias_mitigation.ipynb](03_bias_mitigation.ipynb) mitigates the bias you found.